# Build the actual or forecast Silver demand table

This notebook uses one parser and one consolidation flow for either observed demand or AEMO P5MIN demand forecasts. The `ingestion_type` widget selects the known configuration; every path, field and key is derived internally.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. The notebook reads the selected Bronze CSVs from this volume and writes the corresponding Silver Delta table.

## 1. Select the ingestion type

Choose `actual` to reproduce the existing `time | demand` table, or `forecast` to retain every P5MIN forecast run as `forecast_time | time | demand`. This is the notebook's only external parameter.

In [ ]:
from pyspark.sql import functions as F


dbutils.widgets.dropdown(
    "ingestion_type",
    "actual",
    ["actual", "forecast"],
    "Ingestion type"
)

ingestion_type = dbutils.widgets.get("ingestion_type")

## 2. Derive the internal configuration

Only two configurations are supported. Actual demand keeps the existing Dispatch filters and table name. Forecast demand reads `P5MIN,REGIONSOLUTION`, retains both forecast timestamps and writes a separate table.

In [ ]:
base_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

if ingestion_type == "actual":
    row_pattern = "DISPATCH,REGIONSUM"
    time_column = "SETTLEMENTDATE"
    forecast_time_column = None
    runno_column = "RUNNO"

    source_paths = {
        "monthly": f"{base_path}/monthly_uncompressed/*.CSV",
        "daily": f"{base_path}/daily_uncompressed/*.CSV",
        "current": f"{base_path}/current_uncompressed/*.CSV",
    }

    # Fresher publications may contain corrections, so current data wins ties.
    source_priorities = {
        "monthly": 1,
        "daily": 2,
        "current": 3,
    }

    key_columns = ["time"]
    output_columns = ["time", "demand"]
    output_table = "workspace.default.demand_vic_5min"

elif ingestion_type == "forecast":
    row_pattern = "P5MIN,REGIONSOLUTION"
    time_column = "INTERVAL_DATETIME"
    forecast_time_column = "RUN_DATETIME"
    runno_column = None

    source_paths = {
        "archive": f"{base_path}/forecast/archive_uncompressed/*.CSV",
        "current": f"{base_path}/forecast/current_uncompressed/*.CSV",
    }

    # Current P5MIN files are fresher than the archive, so they win ties.
    source_priorities = {
        "archive": 1,
        "current": 2,
    }

    key_columns = ["forecast_time", "time"]
    output_columns = ["forecast_time", "time", "demand"]
    output_table = "workspace.default.aemo_demand_forecast_vic_5min"

else:
    raise ValueError(
        "ingestion_type must be either 'actual' or 'forecast'"
    )

## 3. Parse the selected AEMO section

AEMO files contain several tables in one CSV. The parser therefore finds the `I` row for the configured section, uses it to locate columns, and keeps only matching `D` rows. It never filters on a generic `I` or `D` prefix.

Both modes normalize the timestamp being described to `time` and `TOTALDEMAND` to `demand`. Forecast mode additionally retains `RUN_DATETIME` as `forecast_time`; the `RUNNO = 1` filter is applied only to actual demand.

In [ ]:
def parse_aemo(path):
    # Read each physical line as text before selecting one table section.
    raw = spark.read.text(path)

    header_row = (
        raw
        .filter(F.col("value").startswith(f"I,{row_pattern}"))
        .select(F.split("value", ",").alias("fields"))
        .first()
    )

    if header_row is None:
        raise ValueError(
            f"No I row found for {row_pattern} in {path}"
        )

    header = header_row["fields"]

    # Resolve positions from the published I row rather than fixed indexes.
    time_i = header.index(time_column)
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    runno_i = (
        header.index(runno_column)
        if runno_column
        else None
    )
    forecast_time_i = (
        header.index(forecast_time_column)
        if forecast_time_column
        else None
    )

    df = (
        raw
        .filter(F.col("value").startswith(f"D,{row_pattern}"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
    )

    # RUNNO exists in DISPATCHREGIONSUM but not in P5MIN REGIONSOLUTION.
    if runno_i is not None:
        df = df.filter(F.col("fields")[runno_i].cast("int") == 1)

    def timestamp_expression(index, alias):
        return F.to_timestamp(
            F.regexp_replace(
                F.col("fields")[index],
                '"',
                ""
            ),
            "yyyy/MM/dd HH:mm:ss"
        ).alias(alias)

    selected_columns = []

    if forecast_time_i is not None:
        selected_columns.append(
            timestamp_expression(forecast_time_i, "forecast_time")
        )

    selected_columns.extend([
        timestamp_expression(time_i, "time"),
        F.col("fields")[demand_i].cast("double").alias("demand"),
    ])

    return df.select(*selected_columns)

## 4. Apply the shared parser to every configured source

The source names and paths change by mode, but loading, parsing and temporary-view creation remain shared.

In [ ]:
source_dfs = {
    source_name: parse_aemo(source_path)
    for source_name, source_path in source_paths.items()
}

for source_name, source_df in source_dfs.items():
    source_df.createOrReplaceTempView(source_name)

## 5. Combine, deduplicate and write Delta with Spark SQL

`UNION ALL` keeps every candidate row. `ROW_NUMBER` then retains one row for the configured natural key. Actual demand uses `time`; forecasts use `forecast_time + time`, preserving different forecast vintages for the same eventual interval.

In [ ]:
selected_sql = ", ".join(output_columns)
partition_sql = ", ".join(key_columns)

union_sql = "\n\nUNION ALL\n\n".join(
    f"SELECT {selected_sql}, {source_priorities[source_name]} AS priority "
    f"FROM {source_name}"
    for source_name in source_paths
)

spark.sql(f'''
CREATE OR REPLACE TABLE {output_table}
USING DELTA
AS

WITH all_data AS (
    {union_sql}
),

deduplicated AS (
    SELECT
        {selected_sql},
        ROW_NUMBER() OVER (
            PARTITION BY {partition_sql}
            ORDER BY priority DESC
        ) AS rn
    FROM all_data
)

SELECT
    {selected_sql}
FROM deduplicated
WHERE rn = 1
''')

## 6. Inspect the selected Silver table

The actual result contains one demand value per timestamp. The forecast result contains the complete sequence of future intervals for every forecast run.

In [ ]:
display(
    spark.table(output_table)
    .orderBy(*key_columns)
)

### Forecast trajectory checks

When forecast mode is selected, the first query confirms that one `forecast_time` retains multiple future `time` values. The second confirms that the same eventual `time` can appear in several forecast vintages. No actual-demand join is performed here.

In [ ]:
if ingestion_type == "forecast":
    display(spark.sql(f'''
        SELECT
            forecast_time,
            COUNT(DISTINCT time) AS future_intervals
        FROM {output_table}
        GROUP BY forecast_time
        ORDER BY forecast_time DESC
        LIMIT 20
    '''))

    display(spark.sql(f'''
        SELECT
            time,
            COUNT(DISTINCT forecast_time) AS forecast_vintages
        FROM {output_table}
        GROUP BY time
        HAVING COUNT(DISTINCT forecast_time) > 1
        ORDER BY time DESC
        LIMIT 20
    '''))